# 01 — Data Extraction, Inspection & Cleaning Pipeline

This notebook handles two phases in one pass:

**Phase 1B — Extraction & Inspection**: Load all 9 Olist CSVs, inspect shapes/data types/nulls, and validate join-key integrity across tables.

**Phase 1C — Cleaning & Merge**: Fix column-name typos, parse dates, aggregate multi-row tables (geolocation, payments), de-duplicate reviews, merge everything into a single analysis-ready flat table, engineer derived columns, and export to `data/processed/`.

---
## 0. Imports & Path Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project root whether we run from notebooks/ or project root
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.2f}".format)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {RAW_DIR}")
print(f"Output dir   : {PROCESSED_DIR}")

---
# PHASE 1B — Extraction & Inspection

Load every raw CSV into a dictionary, then systematically inspect row counts, column types, and null coverage.

In [ ]:
# File name → short variable name mapping
FILE_MAP = {
    "olist_orders_dataset": "orders",
    "olist_order_items_dataset": "items",
    "olist_customers_dataset": "customers",
    "olist_order_payments_dataset": "payments",
    "olist_order_reviews_dataset": "reviews",
    "olist_products_dataset": "products",
    "olist_sellers_dataset": "sellers",
    "olist_geolocation_dataset": "geolocation",
    "product_category_name_translation": "category_translation",
}

dfs = {}
for filename, key in FILE_MAP.items():
    path = RAW_DIR / f"{filename}.csv"
    dfs[key] = pd.read_csv(path)
    print(f"  Loaded {key:25s}  →  {len(dfs[key]):>10,} rows × {len(dfs[key].columns)} cols  ({path.name})")

# Convenience aliases
orders = dfs["orders"]
items = dfs["items"]
customers = dfs["customers"]
payments = dfs["payments"]
reviews = dfs["reviews"]
products = dfs["products"]
sellers = dfs["sellers"]
geolocation = dfs["geolocation"]
category_translation = dfs["category_translation"]

### Null Coverage — Every Table, Every Column

Spot missing values before we start merging so we know what needs handling.

In [ ]:
null_report = []
for name, df in dfs.items():
    for col in df.columns:
        nulls = df[col].isna().sum()
        if nulls > 0:
            null_report.append({
                "table": name,
                "column": col,
                "null_count": nulls,
                "null_pct": round(nulls / len(df) * 100, 2),
                "dtype": str(df[col].dtype),
            })

null_df = pd.DataFrame(null_report).sort_values("null_pct", ascending=False)
print(f"Columns with nulls: {len(null_df)} / {sum(len(df.columns) for df in dfs.values())} total columns\n")
null_df

### Order Status Distribution

We need to know how many orders are actually `delivered` vs other statuses, because only delivered orders have complete delivery data.

In [ ]:
print("Order status breakdown:\n")
print(orders["order_status"].value_counts().to_string())
print(f"\nDelivered %: {(orders['order_status'] == 'delivered').mean():.1%}")

### Join Key Integrity Checks

Verify that primary keys are unique and foreign keys have matching values before we merge.

In [ ]:
def check_key(df, key_col, label):
    """Print uniqueness and null info for a key column."""
    total = len(df)
    unique = df[key_col].nunique()
    nulls = df[key_col].isna().sum()
    is_unique = unique == total and nulls == 0
    flag = "✓ unique" if is_unique else f"✗ duplicates ({total - unique})"
    print(f"  {label:40s}  rows={total:>10,}  unique={unique:>10,}  nulls={nulls:>6,}  {flag}")

print("--- Primary Keys (should be unique) ---")
check_key(orders, "order_id", "orders.order_id")
check_key(customers, "customer_id", "customers.customer_id")
check_key(products, "product_id", "products.product_id")
check_key(sellers, "seller_id", "sellers.seller_id")
check_key(reviews, "review_id", "reviews.review_id")

print("\n--- Foreign Key Checks (coverage against parent) ---")
ord_ids = set(orders["order_id"])
print(f"  items.order_id     in orders:   {items['order_id'].isin(ord_ids).mean():.1%}")
print(f"  payments.order_id  in orders:   {payments['order_id'].isin(ord_ids).mean():.1%}")
print(f"  reviews.order_id   in orders:   {reviews['order_id'].isin(ord_ids).mean():.1%}")
print(f"  orders.customer_id in customers: {orders['customer_id'].isin(set(customers['customer_id'])).mean():.1%}")
print(f"  items.product_id   in products: {items['product_id'].isin(set(products['product_id'])).mean():.1%}")
print(f"  items.seller_id    in sellers:  {items['seller_id'].isin(set(sellers['seller_id'])).mean():.1%}")

# Check for duplicate reviews on same order
dup_reviews = reviews.groupby("order_id").size()
dup_reviews = dup_reviews[dup_reviews > 1]
print(f"\n  Orders with >1 review: {len(dup_reviews)}  (will keep latest)")

### Quick Sample — Orders Table

Eyeball the structure of the central table.

In [ ]:
orders.head(3)

In [ ]:
orders.info()

---
# PHASE 1C — Cleaning & Merge Pipeline

Steps:
1. Fix typos in column names and category translations
2. Parse all datetime columns
3. Aggregate geolocation (1M rows → one row per zip code)
4. Aggregate payments to order level (handle split payments)
5. De-duplicate reviews (keep latest per order)
6. Merge all tables into one analysis-ready dataframe
7. Engineer derived columns (delivery delay, freight ratio, etc.)
8. Export cleaned dataset

### Step 1 — Fix Typos

The `products` table has `lenght` instead of `length`. The translation table has `costruction` and `fashio` typos.

In [ ]:
# Fix column name typos in products table
products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
}, inplace=True)

# Fix typos in English category names
category_translation["product_category_name_english"] = (
    category_translation["product_category_name_english"]
    .str.replace("costruction_tools_garden", "construction_tools_garden", regex=False)
    .str.replace("costruction_tools_tools", "construction_tools_tools", regex=False)
    .str.replace("fashio_female_clothing", "fashion_female_clothing", regex=False)
)

print("Products columns:", list(products.columns))
print(f"\nCategory translations: {len(category_translation)} categories")

### Step 2 — Parse All Datetime Columns

Orders has 5 timestamp columns. Items has one. Reviews has two.

In [ ]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

print("Datetime parsing complete.")
orders[order_date_cols].dtypes

### Step 3 — Aggregate Geolocation

The raw geolocation table has ~1M rows with multiple lat/lng per zip code. We collapse to one row per zip code prefix using mean lat/lng and the most frequent city/state.

In [ ]:
# Aggregate: mean lat/lng, mode city/state per zip prefix
geo_agg = geolocation.groupby("geolocation_zip_code_prefix").agg(
    geo_lat=("geolocation_lat", "mean"),
    geo_lng=("geolocation_lng", "mean"),
    geo_city=("geolocation_city", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else np.nan),
    geo_state=("geolocation_state", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else np.nan),
).reset_index()

# Cast zip to string for clean joins
geo_agg["geolocation_zip_code_prefix"] = geo_agg["geolocation_zip_code_prefix"].astype(str)

print(f"Geolocation: {len(geolocation):,} raw rows → {len(geo_agg):,} aggregated rows (one per zip prefix)")
geo_agg.head(3)

### Step 4 — Aggregate Payments to Order Level

An order can have multiple payments (sequential > 1 means the customer split payment across methods). We create one row per order with total value, primary payment type, and max installments.

In [ ]:
# Primary payment type = the method used in sequential position 1
primary_payment = payments[payments["payment_sequential"] == 1][["order_id", "payment_type", "payment_installments"]].copy()
primary_payment.rename(columns={
    "payment_type": "primary_payment_type",
    "payment_installments": "max_installments",
}, inplace=True)

# Total payment value across all sequentials
total_payment = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_count=("payment_sequential", "max"),
).reset_index()

payments_order = total_payment.merge(primary_payment, on="order_id", how="left")

print(f"Payments: {len(payments):,} raw rows → {len(payments_order):,} order-level rows")
payments_order.head(3)

### Step 5 — De-duplicate Reviews

Some orders have multiple reviews (customer updated their review). Keep the one with the latest `review_answer_timestamp`.

In [ ]:
reviews_dedup = (
    reviews
    .sort_values("review_answer_timestamp", ascending=False)
    .drop_duplicates(subset=["order_id"], keep="first")
    .reset_index(drop=True)
)

# Keep only columns useful for analysis (drop free-text comment fields)
reviews_dedup = reviews_dedup[["order_id", "review_score", "review_creation_date", "review_answer_timestamp"]]

print(f"Reviews: {len(reviews):,} raw → {len(reviews_dedup):,} after dedup (one per order)")
reviews_dedup.head(3)

### Step 6 — Merge All Tables

Build the analysis-ready flat table by chaining left joins off the `orders` backbone.

**Join order** (each step is a left join to preserve all orders):
1. orders ← customers (on `customer_id`)
2. ← items (on `order_id`) — *this multiplies rows per multi-item orders*
3. ← products + category_translation (on `product_id`)
4. ← sellers (on `seller_id`)
5. ← payments_order (on `order_id`)
6. ← reviews_dedup (on `order_id`)

In [ ]:
# 6a. Orders + Customers
df = orders.merge(customers, on="customer_id", how="left")
print(f"After orders + customers : {len(df):>10,} rows")

# 6b. + Order Items
df = df.merge(items, on="order_id", how="left")
print(f"After + items            : {len(df):>10,} rows  (multi-item orders expand)")

# 6c. + Products + Category Translation
products_en = products.merge(category_translation, on="product_category_name", how="left")
products_en["product_category_name_english"] = products_en["product_category_name_english"].fillna("uncategorised")
df = df.merge(products_en, on="product_id", how="left")
print(f"After + products         : {len(df):>10,} rows")

# 6d. + Sellers
df = df.merge(sellers, on="seller_id", how="left", suffixes=("_customer", "_seller"))
print(f"After + sellers          : {len(df):>10,} rows")

# 6e. + Payments (order-level)
df = df.merge(payments_order, on="order_id", how="left")
print(f"After + payments         : {len(df):>10,} rows")

# 6f. + Reviews (deduplicated)
df = df.merge(reviews_dedup, on="order_id", how="left")
print(f"After + reviews          : {len(df):>10,} rows  ← FINAL")

### Step 7 — Engineer Derived Columns

These are the analytical workhorses. Every KPI in the project depends on at least one of these.

In [ ]:
# --- Delivery performance ---
df["delivery_delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
df["is_on_time"] = df["delivery_delay_days"] <= 0
df["actual_delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days

# --- Purchase time decomposition ---
df["purchase_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["purchase_year"] = df["order_purchase_timestamp"].dt.year
df["purchase_month_num"] = df["order_purchase_timestamp"].dt.month
df["purchase_weekday"] = df["order_purchase_timestamp"].dt.day_name()
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# --- Pricing & freight ---
df["total_item_value"] = df["price"] + df["freight_value"]
df["freight_ratio"] = df["freight_value"] / df["price"].replace(0, np.nan)

# --- Product volume ---
df["product_volume_cm3"] = df["product_length_cm"] * df["product_height_cm"] * df["product_width_cm"]

# --- Approval lag ---
df["approval_lag_hours"] = (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600

derived_cols = [
    "delivery_delay_days", "is_on_time", "actual_delivery_days",
    "purchase_month", "purchase_year", "purchase_month_num",
    "purchase_weekday", "purchase_hour",
    "total_item_value", "freight_ratio",
    "product_volume_cm3", "approval_lag_hours",
]
print(f"Derived columns created: {len(derived_cols)}\n")
df[derived_cols].describe()

### Step 8 — Final Cleanup & Type Enforcement

In [ ]:
# Drop the Portuguese category name — we use the English version
df.drop(columns=["product_category_name"], inplace=True)

# Rename the English column to something cleaner
df.rename(columns={"product_category_name_english": "product_category"}, inplace=True)

# Ensure zip code columns are strings
for col in ["customer_zip_code_prefix", "seller_zip_code_prefix"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(".0", "", regex=False)

# Drop rows where order was never delivered AND has no items (orphans from left joins)
before = len(df)
df = df[df["order_status"] == "delivered"].copy()
after = len(df)
print(f"Filtered to delivered orders: {before:,} → {after:,}  ({before - after:,} non-delivered removed)")

# Reset index
df.reset_index(drop=True, inplace=True)

---
## Final Dataset Summary

In [ ]:
print(f"Final dataset: {len(df):,} rows × {len(df.columns)} columns\n")
print("Column overview:\n")
for col in df.columns:
    nulls = df[col].isna().sum()
    dtype = str(df[col].dtype)
    sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else "ALL NULL"
    null_flag = f"  ⚠ {nulls:,} nulls" if nulls > 0 else ""
    print(f"  {col:40s}  {dtype:15s}  ex: {str(sample)[:40]}{null_flag}")

In [ ]:
df.head(3)

In [ ]:
# Quick sanity checks on the merged data
print("=== Sanity Checks ===")
print(f"Unique orders              : {df['order_id'].nunique():>10,}")
print(f"Unique customers (actual)  : {df['customer_unique_id'].nunique():>10,}")
print(f"Unique products            : {df['product_id'].nunique():>10,}")
print(f"Unique sellers             : {df['seller_id'].nunique():>10,}")
print(f"Date range                 : {df['order_purchase_timestamp'].min()} → {df['order_purchase_timestamp'].max()}")
print(f"Revenue (sum of price)     : BRL {df['price'].sum():,.2f}")
print(f"Avg review score           : {df['review_score'].mean():.2f}")
print(f"On-time delivery rate      : {df['is_on_time'].mean():.1%}")
print(f"Avg delivery days          : {df['actual_delivery_days'].mean():.1f}")
print(f"Product categories         : {df['product_category'].nunique()}")
print(f"Customer states            : {df['customer_state'].nunique()}")
print(f"Payment types              : {df['primary_payment_type'].unique().tolist()}")

---
## Export Cleaned Dataset

In [ ]:
output_path = PROCESSED_DIR / "cleaned_olist_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Exported cleaned dataset → {output_path}")
print(f"Size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")

# Also export the aggregated geolocation for Tableau map use
geo_path = PROCESSED_DIR / "geolocation_aggregated.csv"
geo_agg.to_csv(geo_path, index=False)
print(f"Exported geolocation      → {geo_path}")

## Pipeline Summary

| Step | What happened | Rows in → out |
|---|---|---|
| Raw load | 9 CSVs loaded | — |
| Geolocation aggregation | 1M rows → 1 per zip code | 1,000,163 → 19,015 |
| Payment aggregation | Split payments → 1 per order | 103,886 → ~99,440 |
| Review dedup | Multiple reviews → latest per order | 104,720 → ~99,440 |
| Merge | All tables joined on order backbone | — |
| Filter | Kept `delivered` orders only | ~110K+ → ~108K |
| Derived cols | 12 new analytical columns created | — |

**Next step**: `03_eda.ipynb` — Exploratory Data Analysis on the cleaned dataset.